# Cross-Dataset JRN Consolidation: Probe Validity, Join-Selector Diagnostic, and Paper-Ready Tables

This notebook consolidates 8 experiments across 4 RelBench datasets into a comprehensive JRN (Join Reproduction Number) evaluation. It computes:

- **Fisher z meta-analytic probe validity** (weighted rho=0.876, CI [0.821, 0.915])
- **JRN as binary join selector** (ROC-AUC=1.0, Youden threshold)
- **Aggregation sensitivity reanalysis** (flat, rho=0.099)
- **JRN distribution stats** and bimodality analysis
- **Multiplicative compounding** (R²=0.828)
- **7 paper-ready tables** and hypothesis scorecard (3 SUPPORTED, 1 PARTIAL, 1 UNSUPPORTED)

In [ ]:
import subprocess, sys
def _pip(*a): subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', *a])

# No non-Colab packages needed — all imports are pre-installed on Colab

# Core packages: pre-installed on Colab, install locally to match Colab env
if 'google.colab' not in sys.modules:
    _pip('numpy==2.0.2', 'pandas==2.2.2', 'scikit-learn==1.6.1', 'scipy==1.16.3', 'matplotlib==3.10.0', 'tabulate==0.9.0')

In [ ]:
import json
import math
import os
from collections import defaultdict

import numpy as np
from scipy import stats
from scipy.signal import find_peaks
from scipy.stats import gaussian_kde
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

In [ ]:
GITHUB_DATA_URL = "https://raw.githubusercontent.com/AMGrobelnik/ai-invention-bc07ab-join-reproduction-number-epidemiology-in/main/evaluation_iter4_cross_dataset_j/demo/mini_demo_data.json"

def load_data():
    try:
        import urllib.request
        with urllib.request.urlopen(GITHUB_DATA_URL) as response:
            return json.loads(response.read().decode())
    except Exception: pass
    if os.path.exists("mini_demo_data.json"):
        with open("mini_demo_data.json") as f: return json.load(f)
    raise FileNotFoundError("Could not load mini_demo_data.json")

In [ ]:
data = load_data()
print(f"Loaded data with {len(data['datasets'])} datasets")
print(f"Metrics: {len(data['metrics_agg'])} aggregated metrics")
for ds in data['datasets']:
    print(f"  {ds['dataset']}: {len(ds['examples'])} examples")

In [ ]:
# ── Configuration ──
# Number of bootstrap iterations for ROC-AUC confidence intervals
N_BOOTSTRAP = 100  # Original: 1000

## Utility Functions

Helper functions for Fisher z-transform, safe parsing, and data extraction.

In [ ]:
def safe_float(v, default=None):
    """Safely parse float from string or number."""
    if v is None:
        return default
    try:
        return float(v)
    except (ValueError, TypeError):
        return default


def parse_output(output_str):
    """Parse JSON from output field."""
    try:
        return json.loads(output_str)
    except (json.JSONDecodeError, TypeError):
        return {}


def fisher_z(rho):
    """Fisher z-transform: arctanh(rho)."""
    rho = max(-0.9999, min(0.9999, rho))
    return 0.5 * math.log((1 + rho) / (1 - rho))


def inv_fisher_z(z):
    """Inverse Fisher z-transform: tanh(z)."""
    return math.tanh(z)


def get_dataset_examples(data, dataset_name):
    """Get examples from a named dataset in the loaded data."""
    for ds in data['datasets']:
        if ds['dataset'] == dataset_name:
            return ds['examples']
    return []

## Analysis 1: Cross-Dataset Probe Validity (Fisher z Meta-Analysis)

Each dataset provides a Spearman rho measuring how well the JRN probe (lightweight estimator) tracks the full JRN (ground truth). We combine these via Fisher z meta-analysis to get a weighted overall correlation and 95% CI.

In [ ]:
# ── Analysis 1: Cross-Dataset Probe Validity ──
probe_examples = get_dataset_examples(data, 'probe_validity_per_dataset')

probe_datasets = {}
for ex in probe_examples:
    ds_name = ex['metadata_dataset']
    out = parse_output(ex['output'])
    probe_datasets[ds_name] = {
        'rho': out['rho'],
        'pval': out['pval'],
        'n': out['n'],
    }

# Fisher z meta-analysis
valid_datasets = {k: v for k, v in probe_datasets.items() if v['n'] >= 5}
z_vals, weights, se_vals = [], [], []

for ds_name, ds_info in valid_datasets.items():
    rho = ds_info['rho']
    n = ds_info['n']
    z_i = fisher_z(rho)
    w_i = n - 3
    se_i = 1.0 / math.sqrt(max(w_i, 1))
    z_vals.append(z_i)
    weights.append(w_i)
    se_vals.append(se_i)
    print(f"  {ds_name}: rho={rho:.4f}, p={ds_info['pval']:.6f}, n={n}, z={z_i:.4f}")

z_vals = np.array(z_vals)
weights = np.array(weights, dtype=float)
z_bar = np.sum(weights * z_vals) / np.sum(weights)
se_bar = 1.0 / math.sqrt(np.sum(weights))

# 95% CI and back-transform
z_lower = z_bar - 1.96 * se_bar
z_upper = z_bar + 1.96 * se_bar
meta_rho = inv_fisher_z(z_bar)
ci_lower = inv_fisher_z(z_lower)
ci_upper = inv_fisher_z(z_upper)

# Cochran's Q and I²
Q = float(np.sum(weights * (z_vals - z_bar) ** 2))
k = len(valid_datasets)
Q_p = 1.0 - stats.chi2.cdf(Q, k - 1) if k > 1 else 1.0
I_sq = max(0.0, (Q - (k - 1)) / Q * 100) if Q > 0 else 0.0

print(f"\nMeta-analytic rho = {meta_rho:.4f} [{ci_lower:.4f}, {ci_upper:.4f}]")
print(f"Cochran's Q = {Q:.4f}, p = {Q_p:.4f}, I² = {I_sq:.1f}%")

## Analysis 2: JRN as Binary Join Selector

We label each join as "informative" (JRN > 1.01) or not, then evaluate JRN as a binary classifier via ROC-AUC. We also compute the optimal Youden threshold and bootstrap confidence intervals.

In [ ]:
# ── Analysis 2: JRN as Binary Join Selector ──
jrn_examples = get_dataset_examples(data, 'cross_dataset_jrn_measurements')

jrn_values = np.array([ex['eval_jrn_value'] for ex in jrn_examples])
labels = (jrn_values > 1.01).astype(int)
n_positive = int(np.sum(labels))
n_negative = int(len(labels) - n_positive)

print(f"Positive (JRN>1.01): {n_positive}, Negative: {n_negative}")
print(f"Class balance: {n_positive/len(labels):.4f}")

selector_metrics = {}
if n_positive > 0 and n_negative > 0:
    roc_auc = roc_auc_score(labels, jrn_values)
    selector_metrics['eval_jrn_roc_auc'] = round(roc_auc, 6)
    print(f"JRN ROC-AUC: {roc_auc:.4f}")

    # Youden's J for optimal threshold
    thresholds = np.sort(np.unique(jrn_values))
    best_j, best_thresh = -1.0, 1.0
    best_sens, best_spec = 0.0, 0.0
    for t in thresholds:
        pred = (jrn_values >= t).astype(int)
        tp = np.sum((pred == 1) & (labels == 1))
        fn = np.sum((pred == 0) & (labels == 1))
        tn = np.sum((pred == 0) & (labels == 0))
        fp = np.sum((pred == 1) & (labels == 0))
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0
        j = sens + spec - 1
        if j > best_j:
            best_j = j
            best_thresh = float(t)
            best_sens = float(sens)
            best_spec = float(spec)

    print(f"Youden threshold: {best_thresh:.4f} (J={best_j:.4f}, sens={best_sens:.4f}, spec={best_spec:.4f})")

    # Bootstrap 95% CI for ROC-AUC
    rng = np.random.RandomState(42)
    boot_aucs = []
    for _ in range(N_BOOTSTRAP):
        idx = rng.choice(len(labels), size=len(labels), replace=True)
        if len(np.unique(labels[idx])) < 2:
            continue
        try:
            boot_aucs.append(roc_auc_score(labels[idx], jrn_values[idx]))
        except ValueError:
            continue
    if boot_aucs:
        ci_lo = float(np.percentile(boot_aucs, 2.5))
        ci_hi = float(np.percentile(boot_aucs, 97.5))
        print(f"ROC-AUC 95% CI: [{ci_lo:.4f}, {ci_hi:.4f}]")

## Analysis 3: Aggregation Sensitivity Reanalysis

Tests whether JRN correlates with aggregation sensitivity (how much the performance metric changes with different aggregation functions). A flat relationship suggests JRN is robust to aggregation choice.

In [ ]:
# ── Analysis 3: Aggregation Sensitivity ──
sens_examples = get_dataset_examples(data, 'agg_sensitivity_pooled')

all_jrn = [ex['eval_jrn'] for ex in sens_examples]
all_sens = [ex['eval_sensitivity'] for ex in sens_examples]

per_dataset = defaultdict(lambda: {'jrn': [], 'sens': []})
for ex in sens_examples:
    ds_name = ex.get('metadata_dataset', '')
    per_dataset[ds_name]['jrn'].append(ex['eval_jrn'])
    per_dataset[ds_name]['sens'].append(ex['eval_sensitivity'])

if len(all_jrn) >= 5:
    all_jrn_arr = np.array(all_jrn)
    all_sens_arr = np.array(all_sens)

    # Pooled Spearman
    rho, pval = stats.spearmanr(all_jrn_arr, all_sens_arr)
    print(f"Pooled JRN-sensitivity rho = {rho:.4f}, p = {pval:.6f}")

    # Per-dataset Spearman
    for ds_name, ds_data in per_dataset.items():
        if len(ds_data['jrn']) >= 5:
            r, p = stats.spearmanr(ds_data['jrn'], ds_data['sens'])
            print(f"  {ds_name}: rho = {r:.4f}")

    # Sensitivity by JRN tertile
    tertile_edges = np.percentile(all_jrn_arr, [33.33, 66.67])
    low_mask = all_jrn_arr <= tertile_edges[0]
    mid_mask = (all_jrn_arr > tertile_edges[0]) & (all_jrn_arr <= tertile_edges[1])
    high_mask = all_jrn_arr > tertile_edges[1]

    for name, mask in [('low', low_mask), ('mid', mid_mask), ('high', high_mask)]:
        if np.sum(mask) > 0:
            print(f"  Tertile {name}: mean sensitivity = {np.mean(all_sens_arr[mask]):.4f}")

    # Kruskal-Wallis
    groups = [all_sens_arr[low_mask], all_sens_arr[mid_mask], all_sens_arr[high_mask]]
    groups = [g for g in groups if len(g) > 0]
    if len(groups) >= 2:
        h_stat, h_pval = stats.kruskal(*groups)
        print(f"  Kruskal-Wallis H = {h_stat:.4f}, p = {h_pval:.6f}")

    # Verdict
    if abs(rho) < 0.2 and pval > 0.05:
        verdict = "flat"
    elif rho > 0.2 and pval < 0.05:
        verdict = "monotonic_positive"
    elif rho < -0.2 and pval < 0.05:
        verdict = "monotonic_negative"
    else:
        verdict = "flat"
    print(f"  Verdict: {verdict}")

## Analysis 4: JRN Distribution

Analyzes the distribution of all JRN measurements: summary statistics, skewness, kurtosis, bimodality coefficient, KDE modes, and per-dataset breakdowns.

In [ ]:
# ── Analysis 4: JRN Distribution ──
jrn_all = np.array([ex['eval_jrn_value'] for ex in jrn_examples])
n = len(jrn_all)

print(f"JRN Distribution (n={n}):")
print(f"  Mean:   {np.mean(jrn_all):.4f}")
print(f"  Median: {np.median(jrn_all):.4f}")
print(f"  Std:    {np.std(jrn_all):.4f}")
print(f"  Range:  [{np.min(jrn_all):.4f}, {np.max(jrn_all):.4f}]")
print(f"  %% above 1.0: {np.mean(jrn_all > 1.0)*100:.1f}%")
print(f"  %% near 1.0 (0.95-1.05): {np.mean((jrn_all >= 0.95) & (jrn_all <= 1.05))*100:.1f}%")

if n >= 8:
    skew_val = stats.skew(jrn_all)
    kurt_val = stats.kurtosis(jrn_all)
    print(f"  Skewness: {skew_val:.4f}")
    print(f"  Kurtosis: {kurt_val:.4f}")

    # Bimodality coefficient
    denom = kurt_val + 3 * ((n-1)**2) / ((n-2)*(n-3))
    bc = (skew_val**2 + 1) / denom if denom != 0 else 0
    print(f"  Bimodality coefficient: {bc:.4f} (>0.555 suggests bimodality)")

# Shapiro-Wilk
if 3 <= n <= 5000:
    w_stat, w_pval = stats.shapiro(jrn_all)
    print(f"  Shapiro-Wilk: W={w_stat:.4f}, p={w_pval:.6f}")

# KDE modes
if n >= 10:
    jrn_range = np.linspace(float(np.min(jrn_all)) - 0.1, float(np.max(jrn_all)) + 0.1, 500)
    kde = gaussian_kde(jrn_all)
    kde_vals = kde(jrn_range)
    peaks, _ = find_peaks(kde_vals)
    mode_locs = [round(float(jrn_range[p]), 4) for p in peaks]
    print(f"  KDE modes: {len(mode_locs)} at {mode_locs}")

# Per-dataset stats
ds_groups = defaultdict(list)
for ex in jrn_examples:
    ds_groups[ex['metadata_dataset']].append(ex['eval_jrn_value'])

print("\nPer-dataset breakdown:")
for ds_name, vals in sorted(ds_groups.items()):
    arr = np.array(vals)
    print(f"  {ds_name}: n={len(vals)}, mean={np.mean(arr):.4f}, median={np.median(arr):.4f}, std={np.std(arr):.4f}")

## Analysis 5: Multiplicative Compounding

Tests whether JRN compounds multiplicatively along join chains: if join A has JRN=1.2 and join B has JRN=1.3, the chain A→B should have JRN ≈ 1.2×1.3 = 1.56. We compute R² between predicted (product of individual JRNs) and measured chain JRNs.

In [ ]:
# ── Analysis 5: Multiplicative Compounding ──
compound_examples = get_dataset_examples(data, 'compounding_chains')

if len(compound_examples) >= 3:
    predicted = np.array([ex['eval_predicted_chain_jrn'] for ex in compound_examples])
    measured = np.array([ex['eval_measured_chain_jrn'] for ex in compound_examples])

    # R²
    ss_res = np.sum((measured - predicted) ** 2)
    ss_tot = np.sum((measured - np.mean(measured)) ** 2)
    r_squared = 1 - ss_res / ss_tot if ss_tot > 0 else 0.0

    # Mean absolute deviation
    mad = float(np.mean(np.abs(predicted - measured)))

    # Spearman
    rho, pval = stats.spearmanr(predicted, measured)

    # Systematic bias
    bias = float(np.mean(predicted - measured))

    print(f"Multiplicative Compounding (n={len(compound_examples)} chains):")
    print(f"  R² = {r_squared:.4f}")
    print(f"  Mean Absolute Deviation = {mad:.4f}")
    print(f"  Spearman rho = {rho:.4f}")
    print(f"  Systematic bias = {bias:.4f}")

    # Separate by source
    sources = defaultdict(list)
    for ex in compound_examples:
        sources[ex['metadata_source']].append(ex)

    for src, exs in sources.items():
        if len(exs) >= 3:
            pred_s = np.array([e['eval_predicted_chain_jrn'] for e in exs])
            meas_s = np.array([e['eval_measured_chain_jrn'] for e in exs])
            ss_r = np.sum((meas_s - pred_s) ** 2)
            ss_t = np.sum((meas_s - np.mean(meas_s)) ** 2)
            r2_s = 1 - ss_r / ss_t if ss_t > 0 else 0.0
            print(f"  {src}: R² = {r2_s:.4f} (n={len(exs)})")
else:
    print("Insufficient compounding chain data")

## Analysis 6: Hypothesis Scorecard

Evaluates 5 key claims with explicit verdict thresholds (SUPPORTED / PARTIAL / UNSUPPORTED).

In [ ]:
# ── Analysis 6: Hypothesis Scorecard ──
scorecard_examples = get_dataset_examples(data, 'scorecard')

VERDICT_MAP = {3: "SUPPORTED", 2: "PARTIAL", 1: "UNSUPPORTED"}

print("Hypothesis Scorecard:")
print("-" * 50)
for ex in scorecard_examples:
    claim = ex['metadata_claim']
    verdict_code = ex['eval_verdict_code']
    verdict = VERDICT_MAP.get(verdict_code, "UNKNOWN")
    out = parse_output(ex['output'])
    symbol = {"SUPPORTED": "+", "PARTIAL": "~", "UNSUPPORTED": "-"}.get(verdict, "?")
    print(f"  [{symbol}] {claim:20s} → {verdict}")
print("-" * 50)

## Visualization

Four-panel summary: (A) Probe validity forest plot, (B) JRN distribution with KDE, (C) Multiplicative compounding predicted vs measured, (D) Hypothesis scorecard heatmap.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# ── Panel A: Probe Validity Forest Plot ──
ax = axes[0, 0]
ds_names = list(valid_datasets.keys())
rhos = [valid_datasets[d]['rho'] for d in ds_names]
ns = [valid_datasets[d]['n'] for d in ds_names]
# Compute CIs per dataset
ci_lowers = []
ci_uppers = []
for d in ds_names:
    z_i = fisher_z(valid_datasets[d]['rho'])
    se_i = 1.0 / math.sqrt(max(valid_datasets[d]['n'] - 3, 1))
    ci_lowers.append(inv_fisher_z(z_i - 1.96 * se_i))
    ci_uppers.append(inv_fisher_z(z_i + 1.96 * se_i))

y_pos = range(len(ds_names) + 1)
all_rhos = rhos + [meta_rho]
all_lowers = ci_lowers + [ci_lower]
all_uppers = ci_uppers + [ci_upper]
all_names = ds_names + ['Meta-analytic']

colors = ['#4c72b0'] * len(ds_names) + ['#c44e52']
for i, (name, r, lo, hi) in enumerate(zip(all_names, all_rhos, all_lowers, all_uppers)):
    ax.errorbar(r, i, xerr=[[r - lo], [hi - r]], fmt='o', color=colors[i],
                capsize=4, markersize=8 if i < len(ds_names) else 12)
ax.set_yticks(list(y_pos))
ax.set_yticklabels(all_names)
ax.axvline(x=0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Spearman rho')
ax.set_title('(A) Probe Validity: Forest Plot')
ax.set_xlim(-0.1, 1.1)

# ── Panel B: JRN Distribution ──
ax = axes[0, 1]
for ds_name, vals in sorted(ds_groups.items()):
    ax.hist(vals, bins=15, alpha=0.5, label=ds_name)
# KDE overlay
if n >= 10:
    jrn_range_plot = np.linspace(float(np.min(jrn_all)) - 0.1, float(np.max(jrn_all)) + 0.1, 200)
    kde_plot = gaussian_kde(jrn_all)
    ax2 = ax.twinx()
    ax2.plot(jrn_range_plot, kde_plot(jrn_range_plot), 'k-', linewidth=2, label='KDE')
    ax2.set_ylabel('Density')
ax.axvline(x=1.0, color='red', linestyle='--', alpha=0.7, label='JRN=1.0')
ax.set_xlabel('JRN Value')
ax.set_ylabel('Count')
ax.set_title('(B) JRN Distribution Across Datasets')
ax.legend(fontsize=8)

# ── Panel C: Multiplicative Compounding ──
ax = axes[1, 0]
if len(compound_examples) >= 3:
    pred_vals = [ex['eval_predicted_chain_jrn'] for ex in compound_examples]
    meas_vals = [ex['eval_measured_chain_jrn'] for ex in compound_examples]
    src_colors = {}
    for ex in compound_examples:
        src = ex['metadata_source']
        if src not in src_colors:
            src_colors[src] = f'C{len(src_colors)}'

    for ex in compound_examples:
        ax.scatter(ex['eval_predicted_chain_jrn'], ex['eval_measured_chain_jrn'],
                   color=src_colors[ex['metadata_source']], alpha=0.7, s=50)

    # Perfect prediction line
    all_vals = pred_vals + meas_vals
    lims = [min(all_vals) - 0.05, max(all_vals) + 0.05]
    ax.plot(lims, lims, 'k--', alpha=0.5, label='Perfect prediction')
    ax.set_xlim(lims)
    ax.set_ylim(lims)

    # Legend for sources
    for src, color in src_colors.items():
        ax.scatter([], [], color=color, label=src.replace('exp_id2_it2_', '').replace('exp_id2_it3_', ''))
    ax.legend(fontsize=8)

ax.set_xlabel('Predicted Chain JRN')
ax.set_ylabel('Measured Chain JRN')
ax.set_title(f'(C) Multiplicative Compounding (R²={r_squared:.3f})')

# ── Panel D: Hypothesis Scorecard ──
ax = axes[1, 1]
claims = []
verdicts_num = []
verdict_labels = []
for ex in scorecard_examples:
    claims.append(ex['metadata_claim'])
    verdicts_num.append(ex['eval_verdict_code'])
    verdict_labels.append(VERDICT_MAP.get(ex['eval_verdict_code'], '?'))

colors_map = {3: '#2ecc71', 2: '#f39c12', 1: '#e74c3c'}
bar_colors = [colors_map.get(v, 'gray') for v in verdicts_num]
bars = ax.barh(range(len(claims)), verdicts_num, color=bar_colors)
ax.set_yticks(range(len(claims)))
ax.set_yticklabels(claims)
ax.set_xlim(0, 3.5)
ax.set_xticks([1, 2, 3])
ax.set_xticklabels(['UNSUPPORTED', 'PARTIAL', 'SUPPORTED'])
ax.set_title('(D) Hypothesis Scorecard')
for i, (v, label) in enumerate(zip(verdicts_num, verdict_labels)):
    ax.text(v + 0.05, i, label, va='center', fontsize=9)

plt.tight_layout()
plt.savefig('jrn_consolidation_results.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved visualization to jrn_consolidation_results.png")